In [1]:

#1. Setup: Ward Election Panel
# Purpose:
# Set up the notebook and define the election input and
# processed-output locations for the KZN ward panel.


from pathlib import Path
import pandas as pd

# Project structure
PROJECT_ROOT = Path(
    r"C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA"
)

# Input folder containing the already cleaned election files
ELECTION_INPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "KZN_Local_Goverment_elections_2011-21 Cleaned"
)

# Output folder for the ward election panel
ELECTION_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "01_ward_election_panel"
)

ELECTION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Exact input files
ELECTION_FILES = {
    2011: ELECTION_INPUT_DIR / "KZN_2011_clean.csv",
    2016: ELECTION_INPUT_DIR / "KZN_2016_clean.csv",
    2021: ELECTION_INPUT_DIR / "KZN_2021_clean.csv",
}

# Final output
ELECTION_OUTPUT = (
    ELECTION_OUTPUT_DIR
    / "ward_election_panel_2011_2021.csv"
)

print("Setup completed successfully.")
print(f"Project root: {PROJECT_ROOT}")
print(f"Input directory: {ELECTION_INPUT_DIR}")
print(f"Output directory: {ELECTION_OUTPUT_DIR}")
print()

for year, file_path in ELECTION_FILES.items():
    print(f"{year}: {file_path}")

print()
print(f"Output file: {ELECTION_OUTPUT}")

# Basic path verification
assert PROJECT_ROOT.exists(), "Project root does not exist."
assert ELECTION_INPUT_DIR.exists(), "Election input directory does not exist."

for year, file_path in ELECTION_FILES.items():
    assert file_path.exists(), f"{year} election file not found."

print()
print("Path verification: PASSED")

Setup completed successfully.
Project root: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA
Input directory: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\interim\KZN_Local_Goverment_elections_2011-21 Cleaned
Output directory: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\processed\01_ward_election_panel

2011: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\interim\KZN_Local_Goverment_elections_2011-21 Cleaned\KZN_2011_clean.csv
2016: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\interim\KZN_Local_Goverment_elections_2011-21 Cleaned\KZN_2016_clean.csv
2021: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\interim\KZN_Local_Goverment_elections_2011-21 Cleaned\KZN_2021_clean.csv

Output file: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\processed\01_ward_election_panel\ward_election_panel_2011_2021.csv

Path verification: PASSED


In [2]:
#2. — Load and Inspect Election Data
# Purpose:
# Load the three KZN election datasets so we can confirm their
# structure before building the ward-level election panel.


election_data = {}

for year, file_path in ELECTION_FILES.items():
    df = pd.read_csv(file_path)
    election_data[year] = df

    print(f"{year} dataset loaded successfully.")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    print(f"Columns: {df.columns.tolist()}")
    print()

# Confirm all three datasets loaded
assert set(election_data.keys()) == {2011, 2016, 2021}

print("All three election datasets loaded successfully.")
print("Years available:", sorted(election_data.keys()))

2011 dataset loaded successfully.
Rows: 90,500
Columns: 11
Columns: ['Province', 'Municipality', 'Ward', 'VotingDistrict', 'VotingStationName', 'RegisteredVoters', 'BallotType', 'SpoiltVotes', 'PartyName', 'TotalValidVotes', 'DateGenerated']

2016 dataset loaded successfully.
Rows: 117,987
Columns: 11
Columns: ['Province', 'Municipality', 'Ward', 'VotingDistrict', 'VotingStationName', 'RegisteredVoters', 'BallotType', 'SpoiltVotes', 'PartyName', 'TotalValidVotes', 'DateGenerated']

2021 dataset loaded successfully.
Rows: 251,001
Columns: 11
Columns: ['Province', 'Municipality', 'Ward', 'VotingDistrict', 'VotingStationName', 'RegisteredVoters', 'BallotType', 'SpoiltVotes', 'PartyName', 'TotalValidVotes', 'DateGenerated']

All three election datasets loaded successfully.
Years available: [2011, 2016, 2021]


In [3]:
#3. — Check Election Fields
# Purpose:
# Confirm the fields needed to aggregate the election data
# from voting-station and party level to ward level.

required_columns = [
    "Municipality",
    "Ward",
    "VotingDistrict",
    "RegisteredVoters",
    "BallotType",
    "SpoiltVotes",
    "TotalValidVotes",
    "PartyName"
]

for year, df in election_data.items():
    missing_columns = [
        column for column in required_columns
        if column not in df.columns
    ]

    print(f"{year}:")
    print(f"Missing required columns: {missing_columns}")
    print()

    assert not missing_columns, (
        f"{year} is missing required columns: {missing_columns}"
    )

print("Required election fields verification: PASSED")

2011:
Missing required columns: []

2016:
Missing required columns: []

2021:
Missing required columns: []

Required election fields verification: PASSED


In [4]:
#4. — Check Election Grain
# Purpose:
# Check the voting-station, ballot-type and party structure
# before aggregating the datasets to one row per ward per year.

for year, df in election_data.items():
    print(f"{year} election data:")

    print(f"Unique municipalities: {df['Municipality'].nunique():,}")
    print(f"Unique wards: {df['Ward'].nunique():,}")
    print(f"Unique voting districts: {df['VotingDistrict'].nunique():,}")
    print(f"Unique voting stations: {df['VotingStationName'].nunique():,}")
    print(f"Ballot types: {df['BallotType'].dropna().unique().tolist()}")
    print(f"Party records: {df['PartyName'].nunique():,}")
    print()

print("Election grain inspection completed.")

2011 election data:
Unique municipalities: 51
Unique wards: 828
Unique voting districts: 4,358
Unique voting stations: 4,301
Ballot types: ['PR', 'Ward', 'DC 40%']
Party records: 28

2016 election data:
Unique municipalities: 44
Unique wards: 870
Unique voting districts: 4,792
Unique voting stations: 4,719
Ballot types: ['PR', 'Ward', 'DC 40%']
Party records: 39

2021 election data:
Unique municipalities: 44
Unique wards: 901
Unique voting districts: 4,940
Unique voting stations: 4,866
Ballot types: ['PR', 'Ward', 'DC 40%']
Party records: 79

Election grain inspection completed.


In [5]:
#5. — Filter to Ward Ballots
# Purpose:
# Keep only Ward ballot records so the turnout calculation
# represents the ward election rather than other ballot types.

ward_election_data = {}

for year, df in election_data.items():
    ward_df = df[df["BallotType"] == "Ward"].copy()
    ward_election_data[year] = ward_df

    print(f"{year}:")
    print(f"Rows after Ward ballot filter: {len(ward_df):,}")
    print(f"Unique wards: {ward_df['Ward'].nunique():,}")
    print(f"Ballot types remaining: {ward_df['BallotType'].unique().tolist()}")
    print()

    assert not ward_df.empty, f"{year} has no Ward ballot records."
    assert ward_df["BallotType"].nunique() == 1
    assert ward_df["BallotType"].iloc[0] == "Ward"

print("Ward ballot filtering: PASSED")

2011:
Rows after Ward ballot filter: 31,072
Unique wards: 828
Ballot types remaining: ['Ward']

2016:
Rows after Ward ballot filter: 39,546
Unique wards: 870
Ballot types remaining: ['Ward']

2021:
Rows after Ward ballot filter: 82,261
Unique wards: 901
Ballot types remaining: ['Ward']

Ward ballot filtering: PASSED


In [6]:
#6. — Check Registered Voter Duplication
# Purpose:
# Check whether registered voters are repeated across party
# records before calculating ward-level registered voters.

for year, df in ward_election_data.items():
    station_check = (
        df.groupby(
            ["Municipality", "Ward", "VotingDistrict", "VotingStationName"],
            dropna=False
        )["RegisteredVoters"]
        .nunique()
    )

    duplicated_stations = station_check[station_check > 1]

    print(f"{year}:")
    print(f"Voting stations checked: {len(station_check):,}")
    print(f"Stations with different registered-voter values: {len(duplicated_stations):,}")
    print()

print("Registered-voter consistency check completed.")

2011:
Voting stations checked: 4,358
Stations with different registered-voter values: 0

2016:
Voting stations checked: 4,792
Stations with different registered-voter values: 0

2021:
Voting stations checked: 4,940
Stations with different registered-voter values: 0

Registered-voter consistency check completed.


In [7]:
#7. — Aggregate Election Data to Ward Level
# Purpose:
# Aggregate voting-station records to one row per ward per
# election year while keeping registered voters from being counted repeatedly.

ward_panels = []

for year, df in ward_election_data.items():

    # Keep one registered-voter value per voting station
    station_data = (
        df.groupby(
            ["Province", "Municipality", "Ward",
             "VotingDistrict", "VotingStationName"],
            as_index=False
        )
        .agg(
            RegisteredVoters=("RegisteredVoters", "first"),
            SpoiltVotes=("SpoiltVotes", "first"),
            TotalValidVotes=("TotalValidVotes", "sum")
        )
    )

    # Aggregate the voting stations to ward level
    ward_df = (
        station_data
        .groupby(
            ["Province", "Municipality", "Ward"],
            as_index=False
        )
        .agg(
            RegisteredVoters=("RegisteredVoters", "sum"),
            SpoiltVotes=("SpoiltVotes", "sum"),
            TotalValidVotes=("TotalValidVotes", "sum"),
            VotingDistricts=("VotingDistrict", "nunique"),
            VotingStations=("VotingStationName", "nunique")
        )
    )

    ward_df["ElectionYear"] = year

    ward_panels.append(ward_df)

    print(f"{year}:")
    print(f"Ward records created: {len(ward_df):,}")
    print()

print("Ward-level aggregation completed.")

2011:
Ward records created: 828

2016:
Ward records created: 870

2021:
Ward records created: 901

Ward-level aggregation completed.


In [8]:
#8. — Calculate Ward Turnout Rate
# Purpose:
# Calculate turnout at ward level using valid votes cast
# divided by registered voters for each election year.

for ward_df in ward_panels:
    ward_df["TurnoutRate"] = (
        ward_df["TotalValidVotes"]
        / ward_df["RegisteredVoters"]
    ) * 100

    print(f"{ward_df['ElectionYear'].iloc[0]}:")
    print(f"Turnout calculated for {len(ward_df):,} wards")
    print(
        f"Turnout range: "
        f"{ward_df['TurnoutRate'].min():.2f}% - "
        f"{ward_df['TurnoutRate'].max():.2f}%"
    )
    print()

print("Ward turnout calculation completed.")

2011:
Turnout calculated for 828 wards
Turnout range: 35.32% - 89.91%

2016:
Turnout calculated for 870 wards
Turnout range: 36.76% - 78.78%

2021:
Turnout calculated for 901 wards
Turnout range: 16.88% - 73.34%

Ward turnout calculation completed.


In [9]:
#9. — Check Ward Boundary Consistency
# Purpose:
# Identify wards that are present in all three election years
# so the final panel can distinguish consistent and changing wards.

ward_year_counts = (
    pd.concat(
        [
            df[["Municipality", "Ward", "ElectionYear"]]
            for df in ward_panels
        ],
        ignore_index=True
    )
    .drop_duplicates()
    .groupby(["Municipality", "Ward"])["ElectionYear"]
    .nunique()
    .reset_index(name="YearsPresent")
)

ward_year_counts["BoundaryConsistent"] = (
    ward_year_counts["YearsPresent"] == 3
)

print(f"Unique municipality-ward combinations: {len(ward_year_counts):,}")
print(
    f"Present in all three years: "
    f"{ward_year_counts['BoundaryConsistent'].sum():,}"
)
print(
    f"Not present in all three years: "
    f"{(~ward_year_counts['BoundaryConsistent']).sum():,}"
)

print()
print("Boundary consistency check completed.")

Unique municipality-ward combinations: 2,057
Present in all three years: 0
Not present in all three years: 2,057

Boundary consistency check completed.


In [10]:
#10. — Inspect Ward Identifiers Across Years
# Purpose:
# Compare municipality and ward identifiers across years
# before deciding how to define the boundary-consistency flag.

for year, df in zip(
    [2011, 2016, 2021],
    ward_panels
):
    print(f"{year}:")
    print("Sample municipality and ward identifiers:")

    print(
        df[
            ["Municipality", "Ward"]
        ]
        .drop_duplicates()
        .head(10)
        .to_string(index=False)
    )

    print()

print("Ward identifier inspection completed.")

2011:
Sample municipality and ward identifiers:
                  Municipality          Ward
ETH - eThekwini [Durban Metro] Ward 59500001
ETH - eThekwini [Durban Metro] Ward 59500002
ETH - eThekwini [Durban Metro] Ward 59500003
ETH - eThekwini [Durban Metro] Ward 59500004
ETH - eThekwini [Durban Metro] Ward 59500005
ETH - eThekwini [Durban Metro] Ward 59500006
ETH - eThekwini [Durban Metro] Ward 59500007
ETH - eThekwini [Durban Metro] Ward 59500008
ETH - eThekwini [Durban Metro] Ward 59500009
ETH - eThekwini [Durban Metro] Ward 59500010

2016:
Sample municipality and ward identifiers:
   Municipality          Ward
ETH - eThekwini Ward 59500001
ETH - eThekwini Ward 59500002
ETH - eThekwini Ward 59500003
ETH - eThekwini Ward 59500004
ETH - eThekwini Ward 59500005
ETH - eThekwini Ward 59500006
ETH - eThekwini Ward 59500007
ETH - eThekwini Ward 59500008
ETH - eThekwini Ward 59500009
ETH - eThekwini Ward 59500010

2021:
Sample municipality and ward identifiers:
   Municipality          Ward

The ward IDs themselves are consistent, but the municipality name changed between 2011 and 2016:

2011: ETH - eThekwini [Durban Metro]
2016/2021: ETH - eThekwini

So our previous (Municipality, Ward) comparison incorrectly treated the same ward as different.

We should therefore use the Ward ID as the primary boundary identifier, while keeping municipality as descriptive information. This is also why we inspect the identifiers before creating the flag.

In [11]:
#11. — Recheck Ward Boundary Consistency
# Purpose:
# Check ward consistency using the ward identifier itself,
# since municipality names changed between election years.

ward_year_counts = (
    pd.concat(
        [
            df[["Ward", "ElectionYear"]]
            for df in ward_panels
        ],
        ignore_index=True
    )
    .drop_duplicates()
    .groupby("Ward")["ElectionYear"]
    .nunique()
    .reset_index(name="YearsPresent")
)

ward_year_counts["BoundaryConsistent"] = (
    ward_year_counts["YearsPresent"] == 3
)

print(f"Unique ward identifiers: {len(ward_year_counts):,}")
print(
    f"Present in all three years: "
    f"{ward_year_counts['BoundaryConsistent'].sum():,}"
)
print(
    f"Not present in all three years: "
    f"{(~ward_year_counts['BoundaryConsistent']).sum():,}"
)

print()
print("Ward boundary consistency check completed.")

Unique ward identifiers: 1,011
Present in all three years: 717
Not present in all three years: 294

Ward boundary consistency check completed.


In [12]:
#12. — Add Boundary Consistency Flag
# Purpose:
# Add a flag showing whether each ward identifier is present
# across all three election years.

boundary_flags = ward_year_counts[
    ["Ward", "BoundaryConsistent"]
].copy()

ward_panels_flagged = []

for ward_df in ward_panels:
    ward_df = ward_df.merge(
        boundary_flags,
        on="Ward",
        how="left",
        validate="many_to_one"
    )

    ward_panels_flagged.append(ward_df)

    print(
        f"{ward_df['ElectionYear'].iloc[0]}: "
        f"{len(ward_df):,} ward records"
    )

print()
print("Boundary consistency flag added.")
print(
    "Consistent wards:",
    boundary_flags["BoundaryConsistent"].sum()
)

2011: 828 ward records
2016: 870 ward records
2021: 901 ward records

Boundary consistency flag added.
Consistent wards: 717


In [13]:
#13. — Combine Election Years
# Purpose:
# Combine the three ward-level election panels into one
# historical KZN ward election panel.

ward_election_panel = pd.concat(
    ward_panels_flagged,
    ignore_index=True
)

print("Election years combined successfully.")
print(f"Total ward-year records: {len(ward_election_panel):,}")
print(
    f"Unique wards: "
    f"{ward_election_panel['Ward'].nunique():,}"
)
print(
    f"Election years: "
    f"{sorted(ward_election_panel['ElectionYear'].unique())}"
)

print()
print("Combined panel preview:")
display(ward_election_panel.head())

Election years combined successfully.
Total ward-year records: 2,599
Unique wards: 1,011
Election years: [np.int64(2011), np.int64(2016), np.int64(2021)]

Combined panel preview:


,Province,Municipality,Ward,RegisteredVoters,SpoiltVotes,TotalValidVotes,VotingDistricts,VotingStations,ElectionYear,TurnoutRate,BoundaryConsistent
0,KwaZulu-Natal,ETH - eThekwini [Durban Metro],Ward 59500001,15364,141,8678,11,11,2011,56.482687,True
1,KwaZulu-Natal,ETH - eThekwini [Durban Metro],Ward 59500002,14295,228,9610,18,18,2011,67.226303,True
2,KwaZulu-Natal,ETH - eThekwini [Durban Metro],Ward 59500003,18225,164,12368,16,16,2011,67.862826,True
3,KwaZulu-Natal,ETH - eThekwini [Durban Metro],Ward 59500004,16370,364,10768,10,10,2011,65.778864,True
4,KwaZulu-Natal,ETH - eThekwini [Durban Metro],Ward 59500005,14650,577,9712,6,6,2011,66.293515,True


In [14]:
#14. — Validate Ward Election Panel
# Purpose:
# Confirm the final panel has one row per ward-year and that
# the core election measures are complete and valid.

# Check for duplicate ward-year records
duplicate_rows = (
    ward_election_panel
    .duplicated(subset=["Ward", "ElectionYear"])
    .sum()
)

print(f"Duplicate ward-year records: {duplicate_rows:,}")

# Check missing values in core fields
core_columns = [
    "Ward",
    "ElectionYear",
    "RegisteredVoters",
    "SpoiltVotes",
    "TotalValidVotes",
    "TurnoutRate",
    "BoundaryConsistent"
]

missing_values = (
    ward_election_panel[core_columns]
    .isna()
    .sum()
)

print()
print("Missing values in core fields:")
print(missing_values)

# Check turnout range
invalid_turnout = (
    (ward_election_panel["TurnoutRate"] < 0)
    | (ward_election_panel["TurnoutRate"] > 100)
).sum()

print()
print(f"Invalid turnout rates: {invalid_turnout:,}")

# Final assertions
assert duplicate_rows == 0, "Duplicate ward-year records found."
assert missing_values.sum() == 0, "Missing values found in core fields."
assert invalid_turnout == 0, "Invalid turnout rates found."

print()
print("WARD ELECTION PANEL VALIDATION: PASSED")

Duplicate ward-year records: 0

Missing values in core fields:
Ward                  0
ElectionYear          0
RegisteredVoters      0
SpoiltVotes           0
TotalValidVotes       0
TurnoutRate           0
BoundaryConsistent    0
dtype: int64

Invalid turnout rates: 0

WARD ELECTION PANEL VALIDATION: PASSED


In [15]:
#15. — Save Ward Election Panel
# Purpose:
# Save the validated ward-level election panel so it can
# be used by the later data-merging notebooks.

ward_election_panel.to_csv(
    ELECTION_OUTPUT,
    index=False
)

print("Ward election panel saved successfully.")
print(f"Output file: {ELECTION_OUTPUT}")
print(f"Rows saved: {len(ward_election_panel):,}")
print(f"Columns saved: {len(ward_election_panel.columns)}")

# Confirm the file exists
assert ELECTION_OUTPUT.exists(), "Output file was not created."

print("File existence verification: PASSED")

Ward election panel saved successfully.
Output file: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\processed\01_ward_election_panel\ward_election_panel_2011_2021.csv
Rows saved: 2,599
Columns saved: 11
File existence verification: PASSED


In [16]:
#16. — Verify Saved Ward Election Panel
# Purpose:
# Read the saved panel back from disk to confirm the processed
# file is complete and readable for the next merging notebook.

saved_panel = pd.read_csv(ELECTION_OUTPUT)

print("Saved panel loaded successfully.")
print(f"Rows read: {len(saved_panel):,}")
print(f"Columns read: {len(saved_panel.columns)}")
print(f"Election years: {sorted(saved_panel['ElectionYear'].unique())}")
print(f"Unique wards: {saved_panel['Ward'].nunique():,}")

# Verify the saved file matches the panel before saving
assert len(saved_panel) == len(ward_election_panel)
assert list(saved_panel.columns) == list(ward_election_panel.columns)

print()
print("READ-BACK VERIFICATION: PASSED")
print("NOTEBOOK 01 COMPLETE")

Saved panel loaded successfully.
Rows read: 2,599
Columns read: 11
Election years: [np.int64(2011), np.int64(2016), np.int64(2021)]
Unique wards: 1,011

READ-BACK VERIFICATION: PASSED
NOTEBOOK 01 COMPLETE
